In [1]:
using DualNumbers
using ForwardDiff
using LinearAlgebra
using Quadmath

In [55]:
# Define leviCivitaTensor
leviCivitaTensor = zeros(3, 3, 3)
leviCivitaTensor[1, 2, 3] = 1
leviCivitaTensor[2, 3, 1] = 1
leviCivitaTensor[3, 1, 2] = 1
leviCivitaTensor[1, 3, 2] = -1
leviCivitaTensor[2, 1, 3] = -1
leviCivitaTensor[3, 2, 1] = -1
leviCivitaTensor

3×3×3 Array{Float64, 3}:
[:, :, 1] =
 0.0   0.0  0.0
 0.0   0.0  1.0
 0.0  -1.0  0.0

[:, :, 2] =
 0.0  0.0  -1.0
 0.0  0.0   0.0
 1.0  0.0   0.0

[:, :, 3] =
  0.0  1.0  0.0
 -1.0  0.0  0.0
  0.0  0.0  0.0

In [57]:
cross([1, 2, 3], [2, 7, 9])

3-element Vector{Int64}:
 -3
 -3
  3

In [62]:
product = [0, 0, 0]
a = [1, 2, 3]
b = [2, 7, 9]
for i in 1:3
    for j in 1:3
        for k in 1:3
            product[i] += leviCivitaTensor[i, j, k]*a[j]*b[k]
        end
    end
end
product

3-element Vector{Int64}:
 -3
 -3
  3

In [2]:
toRadians::Float128 = pi/180
numberOfAtoms::Int64 = 4
masses::Vector{Float128} = Float128.([12.0, 31.972071, 1.00782505, 1.00782505])
valenceCoordinates = Float128.([1.20337419, 1.10377465, 1.10377465, 2.1265833, 2.1265833, pi])

6-element Vector{Float128}:
 1.20337418999999989921434462303295732e+00
 1.10377465000000007933067536214366555e+00
 1.10377465000000007933067536214366555e+00
 2.12658330000000006521077011711895466e+00
 2.12658330000000006521077011711895466e+00
 3.14159265358979311599796346854418516e+00

In [6]:
function defineMolecularFrameXYZ(valence)
    # Stores type to prevent type conversion problems with duals
    T = eltype(valence) 
    molecularFrameXYZ = zeros(T, numberOfAtoms, 3)
    # C at origin
    # CS defines z-axis
    molecularFrameXYZ[2, 3] = valence[1]
    # CH1
    molecularFrameXYZ[3, 1] = valence[2]*sin(valence[4])*cos(valence[6]/2)
    molecularFrameXYZ[3, 2] = valence[2]*sin(valence[4])*sin(valence[6]/2)
    molecularFrameXYZ[3, 3] = valence[2]*cos(valence[4])
    # CH2
    molecularFrameXYZ[4, 1] = valence[3]*sin(valence[5])*cos(valence[6]/2)
    molecularFrameXYZ[4, 2] = -valence[3]*sin(valence[5])*sin(valence[6]/2)
    molecularFrameXYZ[4, 3] = valence[3]*cos(valence[5])
    
    return molecularFrameXYZ
end

function transformToCOM(molecularFrame)
    COM = masses'*molecularFrame/sum(masses)
    frameCOM = molecularFrame .- COM
    return frameCOM
end

function defineCOMframe(valence)
    molecularFrame = defineMolecularFrameXYZ(valence)
    frameCOM = transformToCOM(molecularFrame)
    return frameCOM
end

COMpositionVectorOfAtom(i) = valence -> defineCOMframe(valence)[i, :]

function computeTMatrix(valence)
    frameCOM = defineCOMframe(valence)
    T = eltype(valence)
    tMatrix = zeros(T, numberOfAtoms*3, numberOfAtoms*3)
    for i in 1:numberOfAtoms
        # Translational part
        tMatrix[i*3-2:i*3, 1:3] = Matrix(1I, 3, 3)
    
        # Rotational Part
        for j in 1:3
            unitVector = Matrix(1I, 3, 3)[j, :]
            tMatrix[i*3-2:i*3, 3 + j] = cross(unitVector, frameCOM[i, :])
        end
        # Vibrational part
        jacobianCOM = ForwardDiff.jacobian(COMpositionVectorOfAtom(i), valence)
        tMatrix[i*3-2:i*3, 7:end] = jacobianCOM
    end
    return tMatrix
end

# Redundant for these purposes - mainly useful for symbolic stuff
function computeSMatrix(valence)
    tMatrix = computeTMatrix(valence)
    sMatrix = inv(tMatrix)
    return sMatrix
end

computeSMatrixRow(i) = valence -> computeSMatrix(valence)[i, :]
computeSMatrixElement(i, j) = valence -> computeSMatrix(valence)[i, j]

# Note that the upper case GMatrix is the contravariant metric tensor! gMatrix is the covariant
function computegMatrix(valence)
    tMatrix = computeTMatrix(valence)
    T = eltype(valence)
    gMatrix = zeros(T, 3*numberOfAtoms, 3*numberOfAtoms)
    for i in 1:3*numberOfAtoms
        for j in 1:3*numberOfAtoms
            for n in 1:numberOfAtoms
                gMatrix[i, j] += dot(tMatrix[n*3-2:n*3, i], tMatrix[n*3-2:n*3, j])*masses[n]
            end          
        end
    end
    return gMatrix
end

function computeGMatrix(valence)
    gMatrix = computegMatrix(valence)
    GMatrix = inv(gMatrix)
    return GMatrix
end

computeGMatrix (generic function with 1 method)

In [7]:
@time GMatrix = computeGMatrix(valenceCoordinates)

  0.819629 seconds (2.02 M allocations: 132.914 MiB, 4.55% gc time, 99.94% compilation time)


12×12 Matrix{Float128}:
  2.17449348669725669808956114215966903e-02  …  -1.69806052534640204181879053327415819e-36
 -1.80244661005046002020034668191093163e-88      2.19237535839444404717834255668020718e-53
 -3.28812113519704206657185601629595106e-87      2.82637257279066908257396252786450691e-52
  4.90962476953334781682505560181684177e-56     -8.66823621909307220102162812644198977e-52
  2.13646055200268864204206909176623655e-36     -2.46023889952856267219698711204398221e-01
 -2.11393414568631816573277089803848115e-71  …   7.32590214032662732672500765328614375e-68
  1.62908112539963827226057815805308416e-53     -1.30290338191914102681287314636646000e-51
  6.50851853168827369054530177517772099e-53     -1.37203902106294081377639772864552183e-17
  8.36077710393900270113100937823749915e-53     -1.37203902106294081377639772864552197e-17
 -2.96973608718345976843523265306654335e-52      2.27851061921695746523270063803833090e-17
 -2.89958000106595021325624090871812054e-52  …   2.278510619216957

In [8]:
GMatrix[1, 1]

2.17449348669725669808956114215966903e-02

In [9]:
1/sum(masses)

2.17449348669725669808956114215966903e-02

In [12]:
GMatrix[6, 6]*33.7152537836138732499014581824370

1.90256073918949070444804727939697268e+01

In [15]:
GMatrix[12, 12]*33.7152537836138732499014581824370

1.05375340032900906382837392842708780e+02

In [15]:
GMatrix[18, 18]

1.63198069852602447241310294825574524e+00

In [53]:
GMatrix[18, 17]

-4.80892875767042428393724713562899731e-02

In [55]:
GMatrix[4, 4]

7.21631984797708460639297779067141536e-02

In [56]:
GMatrix[5, 5]

7.21631984797708460639297779067141416e-02

In [15]:
GMatrix[7, 7]

1.45853204375620977274330193446118750e-01

In [16]:
1/sum(masses)

3.12244206044199514384601315528676773e-02

In [17]:
GMatrix[1, 1]

3.12244206044199514384601315528676773e-02

In [18]:
GMatrix[2, 2]

3.12244206044199514384601315528676773e-02

In [19]:
GMatrix[3, 3]

3.12244206044199514384601315528676773e-02

In [89]:
detgMatrix(valence) = det(computegMatrix(valence))
detGMatrix(valence) = det(computeGMatrix(valence))
gradientgMatrixDet(valence) = ForwardDiff.gradient(detgMatrix, valence)
gradientGMatrixDet(valence) = ForwardDiff.gradient(detGMatrix, valence)
U2gradArgument(valence) = GMatrix[7:end,7:end]*gradientGMatrixDet(valence)/gradientgMatrixDet(valence)
function computePseudopotential(valence)
    gMatrixDet = det(computegMatrix(valence))
    detgMatrix = computeGMatrix(valence)
    gradientg = gradientgMatrixDet(valence)
    U1 = Float128(0.0)
    U2 = sum(ForwardDiff.jacobian(U2gradArgument, valenceCoordinates))*4
    for i in 1:numberOfAtoms*3-6
        for j in 1:numberOfAtoms*3-6
            U1 += GMatrix[i+6, j+6]*gradientg[i]*gradientg[j]
        end
    end
    U1 /= gMatrixDet^2
    U = (U1 + U2)/32
    return U
end

computePseudopotential (generic function with 1 method)

In [75]:
@time gradientgMatrixDet(valenceCoordinates)

  0.012446 seconds (4.26 k allocations: 4.412 MiB)


12-element Vector{Float128}:
  4.33397046553479442289732824836025084e+07
  6.44295843741672465587349819689572885e+07
  5.64957450127940037178609277985357602e+07
  5.64957450127940037178609277985357408e+07
  5.64957450127940037178609277985357408e+07
 -1.00706863183136090149294046430900469e+07
 -1.13897616374674896978785396518071964e+07
 -1.13897616374674896978785396518072012e+07
 -1.13897616374674896978785396518071964e+07
  4.92249204857911366576918624736458607e-28
  1.26217744835361888865876570445245797e-27
 -2.28514867450811933096815738286638151e-27

In [68]:
@time ForwardDiff.gradient(detgMatrix, valenceCoordinates)

  0.013122 seconds (4.26 k allocations: 4.412 MiB)


12-element Vector{Float128}:
  4.33397046553479442289732824836025084e+07
  6.44295843741672465587349819689572885e+07
  5.64957450127940037178609277985357602e+07
  5.64957450127940037178609277985357408e+07
  5.64957450127940037178609277985357408e+07
 -1.00706863183136090149294046430900469e+07
 -1.13897616374674896978785396518071964e+07
 -1.13897616374674896978785396518072012e+07
 -1.13897616374674896978785396518071964e+07
  4.92249204857911366576918624736458607e-28
  1.26217744835361888865876570445245797e-27
 -2.28514867450811933096815738286638151e-27

In [93]:
@time computePseudopotential(valenceCoordinates)

  0.673660 seconds (22.01 k allocations: 116.878 MiB)


1.90508494618583908786559758497174105e+00

In [99]:
valenceCoordinates[12] = 50.0*toRadians
valenceCoordinates[11] = 5.0*toRadians
valenceCoordinates[10] = 5.0*toRadians
valenceCoordinates[1] = valenceCoordinates[1]*1.2 
# valenceCoordinates[10] = 5.0*toRadians
@time computePseudopotential(valenceCoordinates)

  0.672507 seconds (22.01 k allocations: 116.878 MiB)


1.89991005665346107338706137606501703e+00

In [26]:
(33.7152537836138732499014581824370*1.90508494618590139573775017773483936)/2

32.11521122000002

In [27]:
gMatrixDet = det(computegMatrix(valenceCoordinates))
ForwardDiff.jacobian(valence -> GMatrix[7:end,7:end]*ForwardDiff.gradient(valence -> det(computeGMatrix(valence)), valence)/gMatrixDet, valenceCoordinates)

12×12 Matrix{Float128}:
  3.71577081926952743791440032757687393e-15  …  -8.39393367208879420007765749799642879e-49
  5.18280397946015298438918088356908464e-14     -1.16594185249956646524045223160560954e-47
  4.23885952197627674435492787769349327e-14     -2.54607924261543445577351188922698149e-48
  4.23885952197627676978813684110823746e-14     -4.79167366032293485267551099269132797e-48
  4.30279337232901738303567121724677857e-14     -6.30845333302202527678497430031196010e-48
 -1.43867629313614338486862384215118782e-14  …   2.90518476435380570806679031951740925e-48
 -7.23791123768376728727978174569485125e-15      1.65591590392583822634276844503672914e-48
 -7.23791123768376901050499788772978300e-15      1.96863444032474162065880394157194765e-48
 -1.15697445997803432110503364828721104e-14      2.13914848637189653366863450568373292e-48
 -1.54696039065452734588633566043749238e-15     -4.39279088133927169533682851683343671e-50
 -2.67941399391023791016484640196667114e-15  …  -1.776844142426642

In [ ]:
ForwardDiff.gradient(valence -> det(computeGMatrix(valence)), valenceCoordinates)

In [ ]:
ForwardDiff.gradient(valence -> ForwardDiff.gradient(GMatrix[7:end,7:end]*ForwardDiff.gradient(valence -> det(computeGMatrix(valence)), valence)/gMatrixDet), valenceCoordinates)

In [17]:
sMatrix = computeSMatrix(valenceCoordinates)

18×18 Matrix{Float128}:
  3.74693047253039417261521578634412272e-01  …   1.50463276905252801019998276764447447e-36
 -4.42169318976253393645805236663173886e-35     -6.68191177523048911535134116787870148e-52
 -9.02779661431516806119989660586652805e-36      3.14687527020126127975680513789426709e-02
  7.22659690853622254195893694635890494e-35     -4.32761555494170821976187211998600616e-85
 -7.03396008590603549936443804937428505e-01      0.00000000000000000000000000000000000e+00
  1.36649295735455135715608418105062005e-34  …  -8.18318255855741179361432321644802356e-85
  0.00000000000000000000000000000000000e+00      0.00000000000000000000000000000000000e+00
 -1.18390754869659320019587636232877184e-34     -0.00000000000000000000000000000000000e+00
 -4.68975507191425525283364184342959052e-01      6.09289080858463851320834252792000281e-85
 -4.68975507191424965600879523303621575e-01     -2.92434179656194232985574958274269899e-85
  9.37951014382850864005900148339447466e-01  …  -3.467677819783451

In [41]:
U1 = Float128(0.0)
for i in 1:numberOfAtoms
    for j in 1:3
        for k in 1:3
            U1 += sMatrix[3 + j, 3*i - 3 + k]*(sMatrix[3 + j, 3*i - 3 + k] - sMatrix[3 + k, 3*i - 3 + j])/(8*masses[i])
        end
    end
end

In [67]:
U2 = Float128(0.0)
for i in 1:3 # beta in TROVE paper
    sJacobian = ForwardDiff.jacobian(computeSMatrixRow(i+3), valenceCoordinates)
    for j in 1:3 # alpha in TROVE paper
        for k in 1:3 # gamma in TROVE paper
            for l in 1:numberOfAtoms*3-6
                for n in 1:numberOfAtoms
                    U2 -= leviCivitaTensor[i, j, k]*sMatrix[l, n*3 - 3 + k]*sJacobian[n*3 - 3 + j, l]/masses[n]
                end
            end
        end
    end
end
U2 = U2/4
# for i in 1:numberOfAtoms
#     for j in 1:3
#         for k in 1:numberOfAtoms-6
#             unitVector = Matrix(1I, 3, 3)[j, :]
#             cross(unitVector, sMatrix[6+k, i*3-2:i*3])*
#     end
# end

-3.46516204439887547732767308998847473e-35

In [42]:
U1

2.00569581645338610824963188998765609e-01

In [65]:
U2

2.01745895227578223807345509620946169e-02

In [94]:
Dual(Float128(0.0), Float128(1.0))

0.00000000000000000000000000000000000e+00 + 1.00000000000000000000000000000000000e+00ɛ

In [79]:
typeof(Dual(2.0, 1.0))

Dual128 (alias for Dual{Float64})

In [75]:
dualpart(tan(Dual(2, 1)))

5.774399204041917

In [167]:
ForwardDiff.derivative(x -> ForwardDiff.derivative(x -> x^2, x), Float128(2.0))

2.00000000000000000000000000000000000e+00

In [169]:
ForwardDiff.derivative(x -> ForwardDiff.derivative(x -> ForwardDiff.derivative(x -> x^3, x), x), Float128(2.0))

6.00000000000000000000000000000000000e+00